Cell 1: Nạp dữ liệu & Tái tạo đúng tập Validation / Test (60/20/20)

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import linear_kernel
from surprise import Dataset, Reader, SVD, accuracy

# Cấu hình đường dẫn
DATA_DIR = os.path.join("..", "..", "data", "processed")
MODEL_DIR = os.path.join("..", "saved_models")
IMG_DIR = os.path.join("..", "..", "docs", "images")
os.makedirs(IMG_DIR, exist_ok=True)

# 1. Nạp dữ liệu đã làm sạch
ratings_df = pd.read_csv(os.path.join(DATA_DIR, "ratings_cleaned.csv"))
books_df = pd.read_csv(os.path.join(DATA_DIR, "books_cleaned.csv"), low_memory=False)

ratings_df["User-ID"] = ratings_df["User-ID"].astype(int)
ratings_df["ISBN"] = ratings_df["ISBN"].astype(str)
ratings_df["Book-Rating"] = ratings_df["Book-Rating"].astype(float)

# 2. Chia tập 60/20/20 đồng nhất với collaborative_filltering.ipynb
train_val_df, test_df = train_test_split(ratings_df, test_size=0.20, random_state=42, shuffle=True)
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42, shuffle=True)

# 3. Load 3 mô hình đã lưu
svd_model = joblib.load(os.path.join(MODEL_DIR, "collaborative_model.pkl"))
pop_data = joblib.load(os.path.join(MODEL_DIR, "popularity_model.pkl"))
cb_data = joblib.load(os.path.join(MODEL_DIR, "content_based_model.pkl"))

print("✅ Đã nạp xong dữ liệu và 3 mô hình!")

Cell 2: Đánh giá sai số dự đoán điểm (RMSE & MAE) trên tập Test

In [ ]:
# --- 1. Đánh giá Collaborative Filtering (SVD) ---
testset = list(test_df[["User-ID", "ISBN", "Book-Rating"]].itertuples(index=False, name=None))
svd_preds = svd_model.test(testset)
svd_rmse = accuracy.rmse(svd_preds, verbose=False)
svd_mae = accuracy.mae(svd_preds, verbose=False)

# --- 2. Đánh giá Popularity-Based ---
pop_df = pop_data['popular_books_df']
id_col_pop = pop_data['id_col']
pop_score_map = dict(zip(pop_df[id_col_pop].astype(str), pop_df['rating_mean']))
global_mean = train_df["Book-Rating"].mean()

y_true = test_df["Book-Rating"].values
y_pred_pop = test_df["ISBN"].map(pop_score_map).fillna(global_mean).values

pop_rmse = np.sqrt(np.mean((y_true - y_pred_pop) ** 2))
pop_mae = np.mean(np.abs(y_true - y_pred_pop))

print(f"[Collaborative Filtering - SVD] Test RMSE: {svd_rmse:.4f} | Test MAE: {svd_mae:.4f}")
print(f"[Popularity-Based Baseline]    Test RMSE: {pop_rmse:.4f} | Test MAE: {pop_mae:.4f}")

Cell 3: Đánh giá chất lượng xếp hạng gợi ý (Precision@K & Recall@K) cho CẢ 3 MÔ HÌNH

In [ ]:
K = 10
RATING_THRESHOLD = 7.0  # Quy ước sách rating >= 7 là sách user thực sự thích

# Lọc các đánh giá tích cực trên tập Test và Train
test_liked = test_df[test_df["Book-Rating"] >= RATING_THRESHOLD].groupby("User-ID")["ISBN"].apply(set)
train_user_books = train_df.groupby("User-ID")["ISBN"].apply(set).to_dict()
train_best_book = (
    train_df[train_df["Book-Rating"] >= RATING_THRESHOLD]
    .sort_values("Book-Rating", ascending=False)
    .drop_duplicates("User-ID")
    .set_index("User-ID")["ISBN"]
    .to_dict()
)

# Chọn các user xuất hiện ở cả Train và Test (có >= 2 sách thích ở Test)
eval_users = [u for u in test_liked.index if len(test_liked[u]) >= 2 and u in train_best_book][:200]

# Chuẩn bị dữ liệu cho Content-Based
tfidf_matrix = cb_data['tfidf_matrix']
cb_indices = cb_data['indices']
cb_books = cb_data['df_books']
cb_id_col = cb_data['id_col']

# Tập ứng viên (Candidate items) gồm các sách xuất hiện trong tập Test để đánh giá xếp hạng
candidate_isbns = list(test_df["ISBN"].unique())
top_pop_list = pop_df[id_col_pop].astype(str).tolist()

def evaluate_top_k(k=10):
    metrics = {
        "Collaborative Filtering (SVD)": {"p": [], "r": []},
        "Popularity-Based": {"p": [], "r": []},
        "Content-Based (TF-IDF)": {"p": [], "r": []}
    }
    
    for uid in eval_users:
        relevant_items = test_liked[uid]
        seen_items = train_user_books.get(uid, set())
        
        # 1. Gợi ý từ Collaborative Filtering
        unseen_candidates = [isbn for isbn in candidate_isbns if isbn not in seen_items][:500]
        cf_preds = [(isbn, svd_model.predict(uid, isbn).est) for isbn in unseen_candidates]
        cf_top_k = [x[0] for x in sorted(cf_preds, key=lambda x: x[1], reverse=True)[:k]]
        
        # 2. Gợi ý từ Popularity-Based
        pop_top_k = [isbn for isbn in top_pop_list if isbn not in seen_items][:k]
        
        # 3. Gợi ý từ Content-Based (dựa trên sách user thích nhất ở tập Train)
        seed_isbn = train_best_book[uid]
        cb_top_k = []
        if seed_isbn in cb_indices:
            idx = cb_indices[seed_isbn]
            if isinstance(idx, pd.Series):
                idx = idx.iloc[0]
            cos_sim = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()
            top_idx = np.argsort(cos_sim)[::-1][1:k+20]
            for i in top_idx:
                rec_isbn = str(cb_books.iloc[i][cb_id_col])
                if rec_isbn not in seen_items and rec_isbn != seed_isbn:
                    cb_top_k.append(rec_isbn)
                if len(cb_top_k) == k:
                    break
        
        # Tính Precision@K và Recall@K cho từng mô hình
        for name, rec_list in [
            ("Collaborative Filtering (SVD)", cf_top_k),
            ("Popularity-Based", pop_top_k),
            ("Content-Based (TF-IDF)", cb_top_k)
        ]:
            hits = len(set(rec_list) & relevant_items)
            metrics[name]["p"].append(hits / k)
            metrics[name]["r"].append(hits / len(relevant_items))
            
    summary = []
    for name, vals in metrics.items():
        summary.append({
            "Mô hình": name,
            f"Precision@{k}": round(np.mean(vals["p"]), 4),
            f"Recall@{k}": round(np.mean(vals["r"]), 4)
        })
    return pd.DataFrame(summary)

ranking_results_df = evaluate_top_k(k=K)
print(ranking_results_df.to_string(index=False))

Cell 4: Tinh chỉnh siêu tham số (Hyperparameter Tuning) trên tập Validation

In [ ]:
reader = Reader(rating_scale=(ratings_df["Book-Rating"].min(), ratings_df["Book-Rating"].max()))
trainset = Dataset.load_from_df(train_df[["User-ID", "ISBN", "Book-Rating"]], reader).build_full_trainset()
valset = list(val_df[["User-ID", "ISBN", "Book-Rating"]].itertuples(index=False, name=None))

param_grid = [
    {"n_factors": 50,  "n_epochs": 20, "lr_all": 0.005, "reg_all": 0.02}, # Cấu hình gốc của TV2
    {"n_factors": 100, "n_epochs": 25, "lr_all": 0.005, "reg_all": 0.05}, # Tăng số chiều ẩn & regularization
    {"n_factors": 50,  "n_epochs": 30, "lr_all": 0.007, "reg_all": 0.05}, # Tăng epochs & learning rate
]

tuning_results = []
for params in param_grid:
    algo = SVD(**params, random_state=42)
    algo.fit(trainset)
    preds = algo.test(valset)
    val_rmse = accuracy.rmse(preds, verbose=False)
    val_mae = accuracy.mae(preds, verbose=False)
    tuning_results.append({**params, "Val_RMSE": round(val_rmse, 4), "Val_MAE": round(val_mae, 4)})

tuning_df = pd.DataFrame(tuning_results)
print("--- KẾT QUẢ TINH CHỈNH SIÊU THAM SỐ (VALIDATION SET) ---")
print(tuning_df.to_string(index=False))

Cell 5: Vẽ biểu đồ trực quan lưu vào docs/images/ để chèn vào Báo cáo Word

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Biểu đồ 1: So sánh RMSE & MAE (SVD vs Popularity)
err_df = pd.DataFrame({
    "Mô hình": ["Collaborative (SVD)", "Collaborative (SVD)", "Popularity-Based", "Popularity-Based"],
    "Metric": ["RMSE", "MAE", "RMSE", "MAE"],
    "Giá trị": [svd_rmse, svd_mae, pop_rmse, pop_mae]
})
sns.barplot(data=err_df, x="Metric", y="Giá trị", hue="Mô hình", ax=axes[0], palette="Blues_d")
axes[0].set_title("So sánh sai số dự đoán điểm (Thấp hơn = Tốt hơn)", fontweight="bold")
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%.4f", padding=3)

# Biểu đồ 2: So sánh Precision@10 & Recall@10 (Cả 3 mô hình)
rank_melted = ranking_results_df.melt(id_vars="Mô hình", var_name="Metric", value_name="Giá trị")
sns.barplot(data=rank_melted, x="Metric", y="Giá trị", hue="Mô hình", ax=axes[1], palette="viridis")
axes[1].set_title(f"So sánh độ chính xác Top-{K} (Cao hơn = Tốt hơn)", fontweight="bold")
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%.4f", padding=3)

plt.tight_layout()
chart_path = os.path.join(IMG_DIR, "model_evaluation_comparison.png")
plt.savefig(chart_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Đã lưu biểu đồ so sánh tại: {chart_path}")